In [ ]:
import sqlite3
import pandas as pd
import os

'''
The main goal for concat SQL was to find and compare the values of
infiltration against heartbleed within the dataset.

My hypothesis was that infiltration would be more noticeable in packet lenght
or packet size against heartbleed proving that the dataset was not only
imbalanced compare to columns like BENIGN being overtaking to the model
hurting these two lacking solid eviedence for the model to predict 
accrately, but that something like infiltration isnt 
represented well but still highly predictable becuase of packet size.

Conlusion however shows that Infiltration Infiltration   :    36 
avg packet lenght was 161.101292
and Heartbleed       11 avg packet lenght was 1626.602318

Heartbleed is more notitable than infiltration however still
represented lower in the prediction model then infiltration.
'''

'\nCURRENTLY UNDER CONSTRUCTION. THIS FILE IS NOT YET FUNCTIONAL.\n'

In [42]:

path = os.path.expanduser('~/.cache/kagglehub/datasets/chethuhn/network-intrusion-dataset/versions/1')

conn = sqlite3.connect('CIC_IDS_2017.db')

conn.execute("DROP TABLE IF EXISTS network_traffic")
conn.commit()

files = [
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv'
]
dfs = []

for file in files:
    try:
        data = pd.read_csv(os.path.join(path, file))
        data.columns = data.columns.str.strip()
        dfs.append(data)
        data.to_sql('network_traffic', conn, if_exists='append', index=False)
        print(f"Loaded {file} ({len(data)} rows)")
    except Exception as e:
        print(f"Couldnt read {file}, {e} :(")

try:
    data = pd.concat(dfs, ignore_index=True)
    print(f"Concat Sucessful! Total rows: {len(data)}, Total Columns: {len(data.columns)}")
except Exception as e:
    print(f"Concat Failed : {e} :( ")

print(data['Label'].value_counts())

Loaded Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv (288602 rows)
Loaded Monday-WorkingHours.pcap_ISCX.csv (529918 rows)
Loaded Friday-WorkingHours-Morning.pcap_ISCX.csv (191033 rows)
Loaded Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv (286467 rows)
Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv (225745 rows)
Loaded Tuesday-WorkingHours.pcap_ISCX.csv (445909 rows)
Loaded Wednesday-workingHours.pcap_ISCX.csv (692703 rows)
Loaded Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv (170366 rows)
Concat Sucessful! Total rows: 2830743, Total Columns: 79
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force

In [32]:
query = """
    SELECT * 
    FROM network_traffic
    LIMIT 5
"""

result = pd.read_sql_query(query, conn)
print(result.head())
 

   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0                22            166                  1                       1   
1             60148             83                  1                       2   
2               123          99947                  1                       1   
3               123          37017                  1                       1   
4                 0      111161336                147                       0   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                            0                            0   
1                            0                            0   
2                           48                           48   
3                           48                           48   
4                            0                            0   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean  \
0                      0                      0            

In [ ]:
""" 
This query indentifies in all CSV files the total number of each attack/BENIGN.
We can also see in this query and compare to the rest of the project simply how fast
SQL is at larger datasets compared to pandas/jypter notebooks at loading 2.8 million rows.
""" 

label = """
    SELECT `Label`, COUNT(*) as total
    FROM network_traffic
    GROUP BY `Label`
    ORDER BY total DESC"""

label_result = pd.read_sql_query(label, conn)
print(label_result)

                         Label    total
0                       BENIGN  2273097
1                     DoS Hulk   231073
2                     PortScan   158930
3                         DDoS   128027
4                DoS GoldenEye    10293
5                  FTP-Patator     7938
6                  SSH-Patator     5897
7                DoS slowloris     5796
8             DoS Slowhttptest     5499
9                          Bot     1966
10    Web Attack � Brute Force     1507
11            Web Attack � XSS      652
12                Infiltration       36
13  Web Attack � Sql Injection       21
14                  Heartbleed       11


In [34]:
Destination = """ SELECT `Label`, `Destination Port`,COUNT(*) as total
            FROM network_traffic
            GROUP BY `Label`, `Destination Port`
            ORDER BY `Label`, COUNT(*) DESC
        """

pd.set_option('display.max_rows', None)
dest_results = pd.read_sql_query(Destination, conn)
print(dest_results)



                            Label  Destination Port   total
0                          BENIGN                53  957812
1                          BENIGN               443  505470
2                          BENIGN                80  235695
3                          BENIGN               123   23879
4                          BENIGN                22   10801
5                          BENIGN               137    7917
6                          BENIGN               389    6247
7                          BENIGN                88    5421
8                          BENIGN                21    5341
9                          BENIGN               465    3657
10                         BENIGN               139    2684
11                         BENIGN              3268    2407
12                         BENIGN               445    1932
13                         BENIGN                 0    1690
14                         BENIGN               138    1612
15                         BENIGN       

In [35]:
Infiltration_counts = """
    SELECT `Label`, COUNT(*) as total
    FROM network_traffic
    GROUP BY `Label`
    ORDER BY total DESC
"""

pd.set_option('display.max_rows', None)
Infiltration_counts_search = pd.read_sql_query(Infiltration_counts, conn)
print(Infiltration_counts_search)

                         Label    total
0                       BENIGN  2273097
1                     DoS Hulk   231073
2                     PortScan   158930
3                         DDoS   128027
4                DoS GoldenEye    10293
5                  FTP-Patator     7938
6                  SSH-Patator     5897
7                DoS slowloris     5796
8             DoS Slowhttptest     5499
9                          Bot     1966
10    Web Attack � Brute Force     1507
11            Web Attack � XSS      652
12                Infiltration       36
13  Web Attack � Sql Injection       21
14                  Heartbleed       11


In [36]:


print("__________")
Infiltration_problem = """
 SELECT  
        `Label`,
        `Destination Port`,
        COUNT(*) as total_flows,
        AVG(`Flow Duration`) as avg_flow_duration,
        AVG(`Packet Length Mean`) as avg_packet_length,
        AVG(`Flow Bytes/s`) as avg_flow_bytes_per_sec,
        AVG(`Total Fwd Packets`) as avg_fwd_packets,
        AVG(`Total Backward Packets`) as avg_bwd_packets,
        AVG(`Init_Win_bytes_forward`) as avg_init_win_fwd,
        AVG(`Init_Win_bytes_backward`) as avg_init_win_bwd,
        AVG(`Fwd PSH Flags`) as avg_psh_flags,
        AVG(`ACK Flag Count`) as avg_ack_flags
    FROM network_traffic
    WHERE `Label` = 'Infiltration'
    GROUP BY `Label`, `Destination Port`
    ORDER BY total_flows DESC
"""

pd.set_option('display.max_rows', None)
Infiltration_search = pd.read_sql_query(Infiltration_problem, conn)
print(Infiltration_search)


__________
          Label  Destination Port  total_flows  avg_flow_duration  \
0  Infiltration               444           36         78407720.5   

   avg_packet_length  avg_flow_bytes_per_sec  avg_fwd_packets  \
0         161.101292             20188.21903       830.222222   

   avg_bwd_packets  avg_init_win_fwd  avg_init_win_bwd  avg_psh_flags  \
0       829.611111       2111.166667            1732.0       0.555556   

   avg_ack_flags  
0       0.833333  


In [38]:

heartBleed_Comparison = """
 SELECT  
        `Label`,
        `Destination Port`,
        COUNT(*) as total_flows,
        AVG(`Flow Duration`) as avg_flow_duration,
        AVG(`Packet Length Mean`) as avg_packet_length,
        AVG(`Flow Bytes/s`) as avg_flow_bytes_per_sec,
        AVG(`Total Fwd Packets`) as avg_fwd_packets,
        AVG(`Total Backward Packets`) as avg_bwd_packets,
        AVG(`Init_Win_bytes_forward`) as avg_init_win_fwd,
        AVG(`Init_Win_bytes_backward`) as avg_init_win_bwd,
        AVG(`Fwd PSH Flags`) as avg_psh_flags,
        AVG(`ACK Flag Count`) as avg_ack_flags
    FROM network_traffic
    WHERE `Label` = 'Heartbleed'
    GROUP BY `Label`, `Destination Port`
    ORDER BY total_flows DESC
"""
pd.set_option('display.max_rows', None)
heartBleed_Comparison_search = pd.read_sql_query(heartBleed_Comparison, conn)
print(heartBleed_Comparison_search)


        Label  Destination Port  total_flows  avg_flow_duration  \
0  Heartbleed               444           11       1.106797e+08   

   avg_packet_length  avg_flow_bytes_per_sec  avg_fwd_packets  \
0        1626.602318            65902.802173      2583.727273   

   avg_bwd_packets  avg_init_win_fwd  avg_init_win_bwd  avg_psh_flags  \
0      1897.181818       2899.272727        276.454545            0.0   

   avg_ack_flags  
0       0.909091  


In [39]:
heartBleed_Comparison = """
 SELECT  
        `Label`,
        `Destination Port`,
        COUNT(*) as total_flows,
        AVG(`Flow Duration`) as avg_flow_duration,
        AVG(`Packet Length Mean`) as avg_packet_length,
        AVG(`Flow Bytes/s`) as avg_flow_bytes_per_sec,
        AVG(`Total Fwd Packets`) as avg_fwd_packets,
        AVG(`Total Backward Packets`) as avg_bwd_packets,
        AVG(`Init_Win_bytes_forward`) as avg_init_win_fwd,
        AVG(`Init_Win_bytes_backward`) as avg_init_win_bwd,
        AVG(`Fwd PSH Flags`) as avg_psh_flags,
        AVG(`ACK Flag Count`) as avg_ack_flags
    FROM network_traffic
    WHERE `Label` = 'DDoS'
    GROUP BY `Label`, `Destination Port`
    ORDER BY total_flows DESC
"""
pd.set_option('display.max_rows', None)
heartBleed_Comparison_search = pd.read_sql_query(heartBleed_Comparison, conn)
print(heartBleed_Comparison_search)

  Label  Destination Port  total_flows  avg_flow_duration  avg_packet_length  \
0  DDoS                80       128024       1.695586e+07          736.90241   
1  DDoS             27636            1       3.956387e+06            6.00000   
2  DDoS             64869            1       6.184400e+06            6.00000   
3  DDoS             64873            1       6.183343e+06            6.00000   

   avg_flow_bytes_per_sec  avg_fwd_packets  avg_bwd_packets  avg_init_win_fwd  \
0                     inf         4.472521         3.255772       3889.978715   
1                9.099211         1.000000         5.000000        229.000000   
2                6.791281         1.000000         6.000000        229.000000   
3                5.822093         1.000000         5.000000        229.000000   

   avg_init_win_bwd  avg_psh_flags  avg_ack_flags  
0        145.410673            0.0       0.546812  
1          0.000000            0.0       1.000000  
2          0.000000            0.0   